# Github Actions 사용
* [Github Action 사용가이드](https://docs.github.com/ko/actions/get-started/quickstart)
* GitHub의 **GitHub Actions**는 코드 저장소에서 발생하는 이벤트를 기반으로 **자동화된 작업(CI/CD)**을 수행하는 기능
* 개발·배포·테스트·데이터 처리까지 자동화

# 1. Github Actions의 중요 구성요소

```yaml
#샘플 코드 : hi.yaml 워크플로우 파일
name: hi 워크플로우
on: push
jobs:
  hi:
    runs-on: ubuntu-latest
    steps:
      - run: echo "hi !!!"
      - uses: actions/checkout@v4
```


### <font color=red>1) Workflow</font>
* 자동화 전체 정의 파일 (`.github/workflows/*.yml` 또는 `.github/workflows/*.yaml` )

### <font color=red>2) Event:</font><font color=blue> **on** </font>
* 워크플로우 실행 트리거

<span style="display:inline-block;position:left;">
    
| 이벤트               | 설명         |
| ----------------- | ---------- |
| push              | 코드 푸시      |
| pull_request      | PR 생성      |
| schedule          | cron 자동 실행 |
| workflow_dispatch | 수동 실행      |

</span>

### <font color=red>3) Job</font>
* 하나의 작업 단위 (병렬 실행 가능)

### <font color=red>4) Step</font>
* Job 내부의 세부 실행 명령

```yaml
steps:
  - name: Checkout
    uses: actions/checkout@v4
```

## Github Actions : <font color=red> **runs-on** </font>
* **Runner**(실행기)는 워크플로우를 실제로 실행하는 **컴퓨터(환경)**
  > “코드를 대신 실행해주는 서버”

## Runner 종류 (분류)

### 1️⃣ GitHub-hosted Runner

* GitHub가 제공하는 가상머신
* 자동 생성/삭제
* 가장 많이 사용

👉 특징:

* 빠른 시작
* 관리 필요 없음

---

### 2️⃣ Self-hosted Runner
* 참조( <a href="[부록]사내서버를 Github Runner로 사용방법.ipynb"><font color=red>[부록]사내서버를 Github Runner로 사용방법</font></a> )
* 사용자가 직접 운영하는 서버
* 회사 내부 서버, 개인 PC 등
* 더 높은 성능 제공 (유료)
* 대규모 빌드, AI 작업용

👉 특징:

* 커스터마이징 가능
* GPU, 특정 장비 사용 가능

---

# Github-hosted Runner 요약

<span style="display:inline-block;position:left;">
    
| Runner         | OS      | 특징     | 
| -------------- | ------- | ------ | 
| **ubuntu-latest**  | **Linux**   | **기본, 빠름** | 
| windows-latest | Windows | MS 환경  | 
| macos-latest   | macOS   | iOS 필수 | 
| ubuntu-22.04   | Linux   | 안정성    |

</span>

# 실전 예제 (Python CI)

## 폴더 구조

```
myapp-simple/
├── .github/
│   └── workflows/
│       └── pythonci.yaml
└── requirements.txt
```

## 파일명: <font color=blue>requirements.txt</font>

---  

```
flask==3.0.0
pytest==8.1.1
```

---  


## 파일명: <font color=blue>.github\workflows\pythonci.yaml</font>

---  

```yaml
name: Python CI

on: push

jobs:
  test:
    runs-on: ubuntu-latest

    steps:
      - uses: actions/checkout@v4

      - name: Python 설정
        uses: actions/setup-python@v5
        with:
          python-version: "3.10"

      - name: 패키지 설치
        run: pip install -r requirements.txt

      - name: 테스트 실행
        run: pytest
```  
---  

# GitHub Container Registry (GHCR) 사용 방법
실습자료( <a href="[부록]GHCR 샘플 이미지 업로드 실습.ipynb"><font color=red>[부록]GHCR 샘플 이미지 업로드 실습.ipynb</font></a> )

---

## GHCR이란?

```
Docker Hub                    GitHub Container Registry
─────────────                 ────────────────────────
별도 사이트                    GitHub 안에 포함
무료 제한 있음                 GitHub 저장소와 연동
로그인 별도 필요               GITHUB_TOKEN 으로 자동 인증
이미지 관리 분리               코드 + 이미지 한곳에서 관리
```

---

## STEP 1. 토큰 권한 설정

```
GitHub 저장소
→ Settings
→ Actions
→ General
→ Workflow permissions

● Read and write permissions  ← 선택
✅ Save
```

---

## STEP 2. 이미지 빌드 & GHCR에 푸시

### 기본 워크플로우

```yaml
name: Docker 이미지 빌드 & 푸시

on:
  push:
    branches:
      - main

jobs:
  build:
    runs-on: ubuntu-latest

    steps:
      # ─────────────────────────────────────
      # STEP 1. 코드 체크아웃
      # ─────────────────────────────────────
      - uses: actions/checkout@v4

      # ─────────────────────────────────────
      # STEP 2. GHCR 로그인
      # ─────────────────────────────────────
      - name: GHCR 로그인
        uses: docker/login-action@v3
        with:
          registry: ghcr.io
          username: ${{ github.actor }}         # GitHub 사용자명 자동 입력
          password: ${{ secrets.GITHUB_TOKEN }} # 자동 발급 토큰

      # ─────────────────────────────────────
      # STEP 3. 이미지 빌드 & 푸시
      # ─────────────────────────────────────
      - name: 이미지 빌드 & 푸시
        uses: docker/build-push-action@v5
        with:
          context: .
          push: true
          tags: |
            ghcr.io/${{ github.repository_owner }}/myapp:latest
            ghcr.io/${{ github.repository_owner }}/myapp:${{ github.sha }}
```

---

## STEP 3. 태그 전략 (버전 관리)

```yaml
      - name: 이미지 메타데이터 설정
        uses: docker/metadata-action@v5
        id: meta
        with:
          images: ghcr.io/${{ github.repository_owner }}/myapp
          tags: |
            # 브랜치명으로 태그
            type=ref,event=branch

            # PR 번호로 태그
            type=ref,event=pr

            # 릴리즈 태그로 태그
            type=semver,pattern={{version}}      # v1.2.3
            type=semver,pattern={{major}}.{{minor}} # v1.2

            # 최신 커밋 SHA로 태그
            type=sha

      - name: 이미지 빌드 & 푸시
        uses: docker/build-push-action@v5
        with:
          context: .
          push: true
          tags: ${{ steps.meta.outputs.tags }}
```

```
생성되는 태그 예시:
ghcr.io/내계정/myapp:main          ← 브랜치명
ghcr.io/내계정/myapp:v1.2.3        ← 릴리즈 버전
ghcr.io/내계정/myapp:sha-abc1234   ← 커밋 SHA
ghcr.io/내계정/myapp:latest        ← 최신
```

---

## STEP 4. GHCR에서 이미지 Pull

```bash
# 로컬에서 Pull
docker pull ghcr.io/내계정/myapp:latest

# 사내 서버에서 Pull (로그인 필요)
echo ${{ secrets.GITHUB_TOKEN }} | \
docker login ghcr.io -u 내계정 --password-stdin

docker pull ghcr.io/내계정/myapp:latest
```

---

## STEP 5. 빌드 & 배포 통합 파이프라인

```yaml
name: 빌드 & 배포 파이프라인

on:
  push:
    branches:
      - main

jobs:
  # ─────────────────────────────────────
  # JOB 1. 이미지 빌드 & GHCR 푸시
  # ─────────────────────────────────────
  build:
    runs-on: ubuntu-latest

    outputs:
      image_tag: ${{ steps.meta.outputs.version }}  # 다음 job에 태그 전달

    steps:
      - uses: actions/checkout@v4

      - name: GHCR 로그인
        uses: docker/login-action@v3
        with:
          registry: ghcr.io
          username: ${{ github.actor }}
          password: ${{ secrets.GITHUB_TOKEN }}

      - name: 메타데이터 설정
        uses: docker/metadata-action@v5
        id: meta
        with:
          images: ghcr.io/${{ github.repository_owner }}/myapp
          tags: type=sha

      - name: 빌드 & 푸시
        uses: docker/build-push-action@v5
        with:
          context: .
          push: true
          tags: ${{ steps.meta.outputs.tags }}

  # ─────────────────────────────────────
  # JOB 2. 사내 서버에 배포
  # ─────────────────────────────────────
  deploy:
    needs: build          # build job 완료 후 실행
    runs-on: [self-hosted, office-server]

    steps:
      - name: GHCR 로그인
        run: |
          echo ${{ secrets.GITHUB_TOKEN }} | \
          docker login ghcr.io \
            -u ${{ github.actor }} \
            --password-stdin

      - name: 최신 이미지 Pull
        run: |
          docker pull ghcr.io/${{ github.repository_owner }}/myapp:latest

      - name: 컨테이너 재시작
        run: |
          docker compose down
          docker compose up -d
          echo "✅ 배포 완료"

      - name: Slack 알림
        if: success()
        run: |
          curl -X POST https://slack.com/api/chat.postMessage \
            -H "Authorization: Bearer ${{ secrets.SLACK_BOT_TOKEN }}" \
            -H "Content-type: application/json; charset=utf-8" \
            --data "{
              \"channel\": \"${{ secrets.SLACK_CHANNEL_ID }}\",
              \"text\": \"✅ 배포 성공\n🐳 이미지: ghcr.io/${{ github.repository_owner }}/myapp:latest\"
            }"
```

---

## GHCR 이미지 확인 방법

```
GitHub 저장소
→ 오른쪽 사이드바
→ Packages 클릭
→ 이미지 목록 확인

또는
https://github.com/내계정?tab=packages
```

---

## docker-compose.yml 에서 GHCR 이미지 사용

```yaml
services:
  web:
    image: ghcr.io/내계정/myapp:latest   # GHCR 이미지 사용
    ports:
      - "8080:8080"
    restart: always
```

---

## 전체 흐름 요약

```
코드 push (main)
      ↓
JOB 1. 빌드
  코드 체크아웃
      ↓
  GHCR 로그인
      ↓
  Docker 이미지 빌드
      ↓
  GHCR에 푸시 ──────▶ ghcr.io/내계정/myapp:latest
                              ↓
JOB 2. 배포                   │
  사내 서버 Runner             │
      ↓                       │
  GHCR에서 Pull ◀─────────────┘
      ↓
  컨테이너 재시작
      ↓
  Slack 알림 ✅
```

---

## Docker Hub vs GHCR 비교

| 항목 | Docker Hub | GHCR |
|------|-----------|------|
| 무료 저장소 | 1개 (private) | 무제한 |
| 인증 | 별도 토큰 | GITHUB_TOKEN 자동 |
| 저장소 연동 | 별도 설정 | GitHub과 자동 연동 |
| 접근 제어 | 별도 설정 | GitHub 권한 그대로 |
| 이미지 공개 | 설정 필요 | 저장소 공개여부 연동 |


# GitHub Actions 자주 쓰는 actions 10가지

---

## 1. `actions/checkout` — 코드 체크아웃
```yaml
- uses: actions/checkout@v4
  with:
    fetch-depth: 0        # 전체 git 히스토리 (기본값 1)
    ref: main             # 특정 브랜치/커밋 체크아웃
```
> 워크플로우에서 저장소 코드를 사용할 수 있게 가져옴. **거의 모든 job 첫 번째 step**에 사용.

---

## 2. `actions/setup-python` — Python 환경 설정
```yaml
- uses: actions/setup-python@v5
  with:
    python-version: "3.11"
    cache: "pip"          # requirements.txt 기반 자동 캐시
```

---

## 3. `actions/setup-node` — Node.js 환경 설정
```yaml
- uses: actions/setup-node@v4
  with:
    node-version: "20"
    cache: "npm"          # npm / yarn / pnpm 캐시 지원
```

---

## 4. `actions/cache` — 캐시 저장/복원
```yaml
- uses: actions/cache@v4
  with:
    path: ~/.cache/pip
    key: ${{ runner.os }}-pip-${{ hashFiles('requirements.txt') }}
    restore-keys: |
      ${{ runner.os }}-pip-
```
> `setup-python`의 `cache: pip`으로 대체 가능하지만, **커스텀 경로 캐시**가 필요할 때 직접 사용.

---

## 5. `actions/upload-artifact` — 파일 업로드
```yaml
- uses: actions/upload-artifact@v4
  with:
    name: test-results
    path: reports/
    retention-days: 7     # 보관 기간 (기본 90일)
```
> 빌드 결과물, 테스트 리포트 등을 **Actions 탭에서 다운로드** 가능하게 저장.

---

## 6. `actions/download-artifact` — 파일 다운로드
```yaml
# upload한 artifact를 다른 job에서 사용할 때
- uses: actions/download-artifact@v4
  with:
    name: test-results
    path: ./downloaded/
```
> `upload-artifact`와 세트로 사용. **job 간 파일 전달**할 때 필수.

---

## 7. `docker/login-action` — Docker 레지스트리 로그인
```yaml
- uses: docker/login-action@v3
  with:
    registry: ghcr.io                        # GitHub Container Registry
    username: ${{ github.actor }}
    password: ${{ secrets.GITHUB_TOKEN }}
```

---

## 8. `docker/build-push-action` — Docker 이미지 빌드 & 푸시
```yaml
- uses: docker/build-push-action@v5
  with:
    context: .
    push: true
    tags: ghcr.io/myorg/myapp:latest
```
> `docker/login-action` 이후에 사용. CI에서 **이미지 자동 빌드 & 배포** 파이프라인의 핵심.

---

## 9. `actions/github-script` — GitHub API 직접 호출
```yaml
- uses: actions/github-script@v7
  with:
    script: |
      await github.rest.issues.createComment({
        issue_number: context.issue.number,
        owner: context.repo.owner,
        repo: context.repo.repo,
        body: '✅ 테스트 통과!'
      })
```
> PR에 자동 코멘트, 이슈 생성 등 **GitHub API를 JS 코드로** 바로 호출.

---

## 10. `peaceiris/actions-gh-pages` — GitHub Pages 배포
```yaml
- uses: peaceiris/actions-gh-pages@v4
  with:
    github_token: ${{ secrets.GITHUB_TOKEN }}
    publish_dir: ./dist       # 배포할 폴더
    publish_branch: gh-pages  # 배포 브랜치
```
> React, Next.js, 문서 사이트 등을 **gh-pages 브랜치에 자동 배포**.
